In [0]:
%run ./01_setup_volumen

In [0]:
import polars as pl
import pandera as pda
from pandera import Column, Check, DataFrameSchema

countries = ["ES", "MX", "CO", "AR", "CL"]
categories = ["electronics", "fashion", "home", "sports", "books"]

silver_path = settings.silver / "orders" / f"run_date={settings.run_date}" / "part-000.parquet"
silver_df = pl.read_parquet(silver_path)
silver_pd = silver_df.to_pandas()

In [0]:
#silver_pd = silver_pd.dropna(subset=["category", "unit_price", "gross_amount"])

In [0]:
# Importamos las librerías necesarias (asumiendo que pda es un alias de pandera o está mal tipeado en el original)
import json
import pandera as pa
from pandera import DataFrameSchema, Column, Check # Implícito por el uso en el código

# 1. DEFINICIÓN DEL ESQUEMA (El Contrato)
silver_schema = DataFrameSchema(
    {
        # Validaciones por columna individual
        "order_id": Column(int, nullable=False, unique=True), # Obligatorio, entero y sin duplicados
        "customer_id": Column(int, nullable=False),
        "product_id": Column(int, nullable=False),
        
        # Debe ser un string, obligatorio, y existir dentro de la lista 'countries'
        "country": Column(str, nullable=False, checks=Check.isin(countries)),
        "category": Column(str, nullable=False, checks=Check.isin(categories)),
        
        # Solo acepta los valores "new" o "existing"
        "customer_segment": Column(str, nullable=False, checks=Check.isin(["new", "existing"])),
        
        # Valores numéricos: cantidad mayor a 0 (gt), precios mayores o iguales a 0 (ge)
        "quantity": Column(int, nullable=False, checks=Check.gt(0)),
        "unit_price": Column(float, nullable=False, checks=Check.ge(0)),
        "gross_amount": Column(float, nullable=False, checks=Check.ge(0)),
        
        "status": Column(str, nullable=False, checks=Check.isin(["paid", "cancelled", "pending"])),
        
        # El monto pagado puede ser nulo (ej. si está 'pending'), pero si existe debe ser >= 0
        "amount": Column(float, nullable=True, checks=Check.ge(0)),
    },
    
    # 2. VALIDACIONES CRUZADAS (Reglas de negocio complejas)
    checks=[
        Check(
            # Verifica que cantidad * precio unitario cuadre con el monto bruto.
            # .round(2) se usa en ambos lados para evitar errores de precisión de decimales flotantes.
            lambda df: (df["gross_amount"].round(2) == (df["quantity"] * df["unit_price"]).round(2)).all(),
            error="gross_amount debe ser quantity * unit_price"
        )
    ],
    # coerce=True fuerza a Pandas a intentar convertir los datos al tipo correcto antes de validar 
    # (ej. convertir un "1" string a 1 entero)
    coerce=True,
)

# 3. PREPARACIÓN DE DIRECTORIOS DE LOGS
# Crea la ruta de la carpeta basándose en la configuración y la fecha de ejecución (Particionamiento)
quality_out_dir = settings.quality / "silver_orders" / f"run_date={settings.run_date}"

# Crea la carpeta físicamente en el sistema de archivos. 
# parents=True crea carpetas intermedias si no existen. exist_ok=True evita errores si la carpeta ya existe.
quality_out_dir.mkdir(parents=True, exist_ok=True)


# 4. EJECUCIÓN DE LA VALIDACIÓN
try:
    # Intenta validar el dataframe 'silver_pd'. 
    # lazy=True hace que revise TODA la tabla y recolecte todos los errores a la vez.
    silver_schema.validate(silver_pd, lazy=True)
    print("Validación Pandera OK")

# Si encuentra datos sucios, atrapa la excepción SchemaErrors (nota: usualmente es pa.errors.SchemaErrors)
except pda.errors.SchemaErrors as exc:
    # Define la ruta para el archivo de errores
    failure_path = quality_out_dir / "validation_errors.parquet"
    
    # Guarda el reporte detallado de los errores (qué falló, dónde, y el valor) en un archivo Parquet
    exc.failure_cases.to_parquet(failure_path, index=False)
    
    # Muestra en pantalla (Databricks/Jupyter) las primeras 20 filas del reporte de errores
    display(exc.failure_cases.head(20))
    
    # Vuelve a lanzar el error para que el Pipeline general falle y no siga procesando datos basura
    raise

In [0]:
# Mirar exactamente las filas rotas que Pandera encontró
display(silver_pd.loc[[49936, 355, 349]])

In [0]:
# Definimos la ruta exacta que compartiste
ruta_errores = "/Volumes/workspace/quality/retail_lab/silver_orders/run_date=2026-03-30/validation_errors.parquet"

# Usamos el lector de Spark para formato Parquet
df_errores = spark.read.parquet(ruta_errores)

# Para visualizar la tabla interactiva en Databricks:
display(df_errores)

# O si quieres ver solo el esquema (nombres de columnas y tipos):
print(df_errores.count())

In [0]:
silver_pd